# Socioeconomic Bias in U.S. Healthcare Access
### Using CDC BRFSS Survey Data

This notebook investigates whether socioeconomic factors — specifically **income level** and **education level** — are associated with disparities in:
- Health insurance coverage
- Cost-related barriers to care
- Routine preventive checkup rates
- Self-rated general health

**Dataset:** [CDC Behavioral Risk Factor Surveillance System (BRFSS)](https://www.cdc.gov/brfss/annual_data/annual_data.htm)  
**Methods:** Descriptive statistics, chi-square tests, logistic regression, correlation analysis

## 0. Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.preprocess import load_and_clean
from src.analysis   import run_all_analyses
from src.visualize  import generate_all_figures, set_style

set_style()
print('Setup complete.')

## 1. Load & Clean Data

> **Download BRFSS data first:** See `data/README.md` for instructions.  
> Update `DATA_PATH` below to point to your downloaded CSV.

In [ ]:
DATA_PATH = '../data/LLCP2022.csv'   # ← update this path

df = load_and_clean(DATA_PATH)
print(f'\nFinal dataset: {df.shape[0]:,} respondents, {df.shape[1]} columns')
df.head(3)

In [ ]:
# Quick overview of key variables
key_cols = ['income_level','education_level','has_insurance','cost_barrier','recent_checkup','general_health']
df[key_cols].describe().round(2)

In [ ]:
# Distribution of income levels in sample
df['income_label'].value_counts().sort_index()

## 2. Run All Analyses

In [ ]:
results = run_all_analyses(df)

## 3. Descriptive Statistics

In [ ]:
print('=== Healthcare Access by Income Level ===')
display_cols = ['income_label','n','insurance_pct','cost_barrier_pct','recent_checkup_pct','mean_health_score']
results['income_summary'][display_cols].style.background_gradient(cmap='Blues', subset=['insurance_pct']) \
       .background_gradient(cmap='Reds_r', subset=['cost_barrier_pct']) \
       .format({'mean_health_score': '{:.2f}', 'n': '{:,.0f}'})

In [ ]:
print('=== Healthcare Access by Education Level ===')
display_cols = ['education_label','n','insurance_pct','cost_barrier_pct','recent_checkup_pct','mean_health_score']
results['education_summary'][display_cols].style.background_gradient(cmap='Blues', subset=['insurance_pct']) \
       .background_gradient(cmap='Reds_r', subset=['cost_barrier_pct']) \
       .format({'mean_health_score': '{:.2f}', 'n': '{:,.0f}'})

## 4. Disparity Gaps

In [ ]:
gaps = {
    'Insurance coverage (by income)':    results['insurance_gap_income'],
    'Cost barrier (by income)':          results['cost_barrier_gap_income'],
    'Insurance coverage (by education)': results['insurance_gap_edu'],
}

for label, gap in gaps.items():
    print(f'\n{label}')
    print(f"  Best-off:  {gap['best_group']}  → {gap['best_val']}%")
    print(f"  Worst-off: {gap['worst_group']} → {gap['worst_val']}%")
    print(f"  Disparity gap: {gap['gap']} percentage points")

## 5. Statistical Tests (Chi-Square)

In [ ]:
chi2_tests = {
    'Income vs Insurance Coverage':    results['chi2_income_insurance'],
    'Income vs Cost Barrier':          results['chi2_income_cost'],
    'Education vs Insurance Coverage': results['chi2_education_insurance'],
}

for name, test in chi2_tests.items():
    sig = '✅ Significant' if test['significant'] else '❌ Not significant'
    print(f'{name}')
    print(f"  χ²={test['chi2']}, df={test['dof']}, p={test['p_value']:.2e}  {sig}")
    print(f"  → {test['interpretation']}\n")

## 6. Logistic Regression — Odds Ratios

In [ ]:
print('=== Insurance Coverage Model ===')
print(f"McFadden's R²: {results['logit_insurance']['pseudo_r2']}")
print(f"N observations: {results['logit_insurance']['n_obs']:,}\n")
display(results['logit_insurance']['summary_df'])

In [ ]:
print('=== Cost Barrier Model ===')
print(f"McFadden's R²: {results['logit_cost_barrier']['pseudo_r2']}")
print(f"N observations: {results['logit_cost_barrier']['n_obs']:,}\n")
display(results['logit_cost_barrier']['summary_df'])

## 7. Correlation Matrix

In [ ]:
results['correlation_matrix']

## 8. Visualizations

In [ ]:
# Generate and save all figures to outputs/
figure_paths = generate_all_figures(results)

# Display inline
for path in figure_paths:
    img = mpimg.imread(path)
    plt.figure(figsize=(11, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

## 9. Key Findings Summary

*(Fill in with your observed numbers after running the analysis)*

### Insurance Coverage
- Respondents earning **>$75k** had ~**X%** insurance coverage vs ~**Y%** for those earning **<$10k** — a **Z percentage point disparity**.
- Chi-square test confirmed this association is statistically significant (p < 0.001).

### Cost Barriers
- Low-income respondents (**<$10k**) were **X× more likely** to report skipping a doctor visit due to cost compared to the highest income group.
- Logistic regression: each income bracket increase reduces the odds of a cost barrier by ~**X%** (OR < 1).

### Education
- College graduates had ~**X%** insurance coverage vs ~**Y%** for those with grades 1–8 education.

### Implications for Medical Bias Research
- These access disparities mean lower-SES populations are **underrepresented in clinical encounters**, which likely propagates downstream bias in medical datasets and guidelines.
- Datasets built from clinical records will systematically underrepresent the sickest, lowest-income patients — those who couldn't afford to show up.